# Task 1: Image to FEN

The model gets one image of a chess board and must write the position in FEN (Forsyth-Edwards Notation). The main score is the exact match on the board part of the FEN. Side to move, castling rights and move counters cannot be seen in the image, so the full-string exact match is at most 3% for every model.

This notebook runs every experiment of Task 1 with Qwen3.5-0.8B: the vanilla model, LoRA
fine-tuning in raster order, training-free and fine-tuned token reordering, and learned reordering
(REOrder). It is the record of our runs on Google Colab with an L4 GPU.

**About the outputs.** Every training run and every evaluation is saved on the Hugging Face Hub
(`bdatm-project` organization). When a cell finds a finished training run or a saved evaluation
on the Hub, it loads it instead of computing it again, so many outputs below show "Using cached
evaluation" instead of a progress bar. Running this notebook needs a Hugging Face token with write
access to the organization. **To check our results without a token, use
`Reproduce_results.ipynb`**, which evaluates the published models again from scratch.

**Sections**

1. Setup
2. Data
3. Experiment settings
4. Vanilla model
5. LoRA baseline (raster order)
6. Training-free reordering (TF)
7. Fine-tuned reordering (SFT)
8. Learned reordering (REOrder)
9. Resolution ablation
10. Results table
11. Diagnostics

## 1. Setup

### 1.1 Packages and imports

Installs the packages and imports the libraries. This cell keeps Colab's preinstalled `torch` and
`torchvision`, like `Task_2.ipynb`, `Task_3.ipynb` and `Reproduce_results.ipynb`, and installs the
main libraries at the versions pinned in `requirements.txt`, so all three tasks are evaluated in
the same environment. Colab's `torchao` is removed because it is too old for `peft`. The cell
prints the `torch` version it runs with.

In [8]:
# Colab's preinstalled torch and torchvision (as in Task 2, Task 3 and Reproduce_results); the main
# libraries at the versions of requirements.txt
!pip install -q "pillow<11.0.0" "transformers==5.20.2" "accelerate==1.15.0" "peft==0.21.2" "datasets==5.1.0" "huggingface_hub==2.2.0" scikit-learn tqdm chess cairosvg python-Levenshtein python-dotenv
!pip uninstall -y torchao -q
!pip install -q "flash-linear-attention==0.5.2"

import gc
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, TaskType
from transformers import AutoModelForImageTextToText, AutoProcessor

print(f"torch {torch.__version__}")

torch 2.11.0+cu130


### 1.2 Repository and project modules

On Colab the repository is cloned from GitHub (or updated, if it is already there) from the branch
in `CONFIG["branch"]`. To run locally, set `CONFIG["colab"] = False` and start Jupyter from the
`notebooks/` folder. The project modules in `src/` are then imported.

In [9]:
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    if repo_root.exists():
        print(f"Repository directory already exists at: {repo_root}; pulling latest changes...")
        pull = subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], capture_output=True, text=True)
        print((pull.stdout or pull.stderr).strip())
    else:
        repo_url = f"https://github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent

print(f"Setup Complete. REPO_ROOT: {repo_root}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

sys.path.insert(0, str(repo_root / "src"))

from eval.results import (
    load_or_init_results_table, append_or_replace, load_results_table, push_results_table, evaluate_with_order,
)
from eval.diagnostics import (
    headline_table, fen_error_breakdown, training_health_report, backfill_cached_metrics, save_diagnostics,
)
from training.finetune import finetune_and_push_chessboard_model, experiment_repo_id, experiment_model_name
from training.learned_reordering import (
    train_with_learned_reordering, finetune_with_frozen_policy, describe_order, greedy_order,
)
from training.reordering import resolve_square_order
from training.resolution import configure_image_resolution
from training.token_reordering import verify_token_reordering

print("All custom modules and eval utilities imported successfully!")

Repository directory already exists at: /content/BigDataAndTextMiningProject; pulling latest changes...
Updating f0e7e2d..f207dd9
Fast-forward
 notebooks/Task_1.ipynb | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: NVIDIA L4
All custom modules and eval utilities imported successfully!


## 2. Data

Logs in to Hugging Face with the token saved as the Colab secret `HF_TOKEN` (locally: in a `.env`
file in the project folder) and loads the Task 1 dataset from
[bdatm-project/dataset_task1](https://huggingface.co/datasets/bdatm-project/dataset_task1):
3,200 train, 400 validation and 400 test samples.

In [10]:
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment, falling back to interactive login.")
    login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task1 = load_dataset(dataset_name, num_proc=16)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

Logged in to Hugging Face Hub using HF_TOKEN.


Resolving data files:   0%|          | 0/3201 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/401 [00:00<?, ?it/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 3200
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 400
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': 'drw7g', 'task': 'task1', 'fen': 'r3k2r/pppq2pp/2nb1p2/1B1Q3P/8/1P2P2P/PBPP1P2/R3K2R w KQkq - 1 13', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Image: The visual representation of the chessboard.\nOutput Format:\nReturn only the valid FEN string representing the position of a

## 3. Experiment settings

**Token-level patch reordering (REOrder).** A scan order is applied to the visual *tokens* inside the model: each image's token embeddings and their M-RoPE positions are permuted together (`src/training/token_reordering.py`), so the board image is never altered and every token keeps its true 2D position. Only the order in which the LLM reads them changes, which matters here because 18 of Qwen3.5's 24 LLM layers are Gated DeltaNet (a recurrent scan with no positional encoding).

The first runs of this notebook rearranged *pixel tiles* instead, which changed the board itself (a "file_wise" image was a transposed board). Their results stay in the results table, labeled `legacy: pixel-tile reordering, 512px`.

**Resolution (patch alignment).** At `IMAGE_SIZE = 256` the LLM sees an 8x8 token grid: exactly one token per chess square. The legacy 512px raster run is the resolution ablation (512px = 2x2 tokens per square).

The next cell sets the image size, the scan orders compared with raster, the number of few-shot
examples, the LoRA configuration and `FORCE_EVAL`, and prints each scan order as a board diagram
(the number on each square is the step at which it is read).

**Evaluating again.** Every evaluation is cached on the Hub: when a model has already been
evaluated, its saved results are loaded instead of running inference. With `FORCE_EVAL = True`
every model is evaluated again and its saved results on the Hub are replaced (the old ones stay in
the history of the results repository). Training is not affected: the finished runs are loaded
from the Hub, so nothing is retrained.

**Running at 512px.** To run this notebook at 512px (2x2 visual tokens per square), set
`IMAGE_SIZE = 512` in the next cell; nothing else needs to change. The tag of every run then
becomes `tok512`, so the models are trained and saved in new repositories
(`qwen-taskN-<order>-tok512-lora`) and their rows get the `[tok512]` suffix in the results table,
without overwriting the 256px results. This needs a Hugging Face token with write access, and
training takes much longer, because each image gives 4 times as many visual tokens. To train only
the raster model of the resolution ablation, stop after Section 5. We did not rerun this notebook
at 512px: our 512px raster models come from an earlier version of the code (see Section 9), and
they are evaluated again in `Reproduce_results.ipynb`.

In [11]:
IMAGE_SIZE = 256   # 256 -> 1 LLM token per square; 512 -> 2x2 tokens per square
TAG = f"tok{IMAGE_SIZE}"   # suffix of every Hub repo / results row of this setup
# compared with the raster baseline ("rank_wise" is raster itself); "hilbert" is from REOrder
STRATEGIES = ["zigzag", "spiral_center_out", "file_wise", "hilbert"]
N_SHOT = 3   # few-shot examples for the vanilla model, drawn from train
TASK = "task1"
model_id = "Qwen/Qwen3.5-0.8B"
hf_org_prefix = "bdatm-project"
# True: run inference again for every model and replace its cached results on the Hub (no retraining)
FORCE_EVAL = False

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

def load_base_model():
    return AutoModelForImageTextToText.from_pretrained(
        model_id, dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32, device_map="auto"
    )

def free_gpu():
    """Call after `del`-ing the model variables, so their GPU memory is released."""
    gc.collect()
    torch.cuda.empty_cache()

for s in ["raster", *STRATEGIES]:
    print(f"{s} (reading step of each square):\n{describe_order(resolve_square_order(s))}\n")

raster (reading step of each square):
8  0  1  2  3  4  5  6  7
7  8  9 10 11 12 13 14 15
6 16 17 18 19 20 21 22 23
5 24 25 26 27 28 29 30 31
4 32 33 34 35 36 37 38 39
3 40 41 42 43 44 45 46 47
2 48 49 50 51 52 53 54 55
1 56 57 58 59 60 61 62 63
   a  b  c  d  e  f  g  h

zigzag (reading step of each square):
8  0  1  2  3  4  5  6  7
7 15 14 13 12 11 10  9  8
6 16 17 18 19 20 21 22 23
5 31 30 29 28 27 26 25 24
4 32 33 34 35 36 37 38 39
3 47 46 45 44 43 42 41 40
2 48 49 50 51 52 53 54 55
1 63 62 61 60 59 58 57 56
   a  b  c  d  e  f  g  h

spiral_center_out (reading step of each square):
8 63 62 61 60 59 58 57 56
7 36 35 34 33 32 31 30 55
6 37 16 15 14 13 12 29 54
5 38 17  4  3  2 11 28 53
4 39 18  5  0  1 10 27 52
3 40 19  6  7  8  9 26 51
2 41 20 21 22 23 24 25 50
1 42 43 44 45 46 47 48 49
   a  b  c  d  e  f  g  h

file_wise (reading step of each square):
8  0  8 16 24 32 40 48 56
7  1  9 17 25 33 41 49 57
6  2 10 18 26 34 42 50 58
5  3 11 19 27 35 43 51 59
4  4 12 20 28 36 44 52 60

## 4. Vanilla model

The processor is set to `IMAGE_SIZE` once here; training, evaluation and generation all go through it. `verify_token_reordering` checks on this environment's `transformers` version that no-order output equals the unpatched model, that embeddings and positions move together, and that the order actually changes the output.

In [12]:
model_vanilla = load_base_model()
processor = AutoProcessor.from_pretrained(model_id)
configure_image_resolution(processor, IMAGE_SIZE)
verify_token_reordering(model_vanilla, processor, dataset_task1["test"][0]["image"], resolve_square_order("zigzag"))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[resolution] processor set to 256px: {'image_size': 256, 'patch_grid': (16, 16), 'llm_token_grid': (8, 8), 'tokens_per_square': 1}
[token reordering] OK: {'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.0625}


{'image_tokens': 64, 'token_grid': (8, 8), 'max_logit_change': 2.0625}

In [13]:
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model_vanilla.device)

print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=128)

trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

Sample ID: sample_000000
FEN: 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33
Prompt provided to the model:
You are a specialized model for chessboard understanding.
Your goal is to extract the exact board state from the provided chessboard image.
Input:
- Board Image: The visual representation of the chessboard.
Output Format:
Return only the valid FEN string representing the position of all pieces on the board.

Real FEN (Ground Truth): 6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - - 0 33

Generating zero-shot prediction...
Predicted FEN (Zero-Shot): e2r1qpppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppppp


Zero-shot and few-shot evaluation of the vanilla model. Few-shot prepends `N_SHOT` solved train examples as earlier chat turns.

In [14]:
all_models_results = load_or_init_results_table(task=TASK, hf_org_prefix=hf_org_prefix)

vanilla_results_df, vanilla_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name("Vanilla Qwen3.5-0.8B (Zero-Shot)", TAG), force=FORCE_EVAL,
)
all_models_results = append_or_replace(all_models_results, vanilla_summary_df)

few_shot_examples = [dataset_task1["train"][i] for i in range(N_SHOT)]
fewshot_results_df, fewshot_summary_df = evaluate_with_order(
    TASK, model_vanilla, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix,
    model_name=experiment_model_name(f"Vanilla Qwen3.5-0.8B ({N_SHOT}-Shot)", TAG),
    few_shot_examples=few_shot_examples, force=FORCE_EVAL,
)
all_models_results = append_or_replace(all_models_results, fewshot_summary_df)
display(fewshot_results_df.head() if fewshot_results_df is not None else fewshot_summary_df)
display(headline_table(all_models_results, TASK))
del model_vanilla
free_gpu()

Loaded existing comparison table for task1 from 'bdatm-project/evaluation-results-task1' (22 model(s) already evaluated).
[task1] Using cached evaluation for 'Vanilla Qwen3.5-0.8B (Zero-Shot) [tok256]', skipping inference.


(…).5-0.8b_%28zero-shot%29_%5Btok256%5D.csv:   0%|          | 0.00/197k [00:00<?, ?B/s]

[task1] Using cached evaluation for 'Vanilla Qwen3.5-0.8B (3-Shot) [tok256]', skipping inference.


(…)en3.5-0.8b_%283-shot%29_%5Btok256%5D.csv:   0%|          | 0.00/96.8k [00:00<?, ?B/s]

,sample_id,ground_truth,predicted,raw_output,fen_exact_match,levenshtein_distance,character_error_rate,square_by_square_accuracy,board_exact_match,board_character_error_rate,side_to_move_match,castling_match,en_passant_match,clocks_match
0,sample_000000,6R1/kbqn1p2/1p5p/1Pp5/4Pb2/7Q/1PPr3P/4K3 b - -...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,0.0,40,0.784314,0.0,0,0.825000,0,0,1,0
1,sample_000001,r1r5/p6n/4pq2/3P1p1k/7P/5PQ1/P5P1/R3R1K1 w - -...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1PP/R7K2R ...,0.0,40,0.784314,0.0,0,0.825000,1,0,1,0
2,sample_000002,8/6pk/5np1/1p5p/p1pP3q/P1P1Q3/1P3P2/5RK1 w - -...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1P/PPP1N2 ...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1P/PPP1N2 ...,0.0,34,0.666667,0.0,0,0.800000,1,1,1,0
3,sample_000003,r4r1k/6p1/p5P1/1p2ppNP/1q2b2Q/4n3/PPPRB3/1K4R1...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1P/PPP1PPP...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1P/PPP1PPP...,0.0,39,0.684211,0.0,0,0.782609,1,1,1,0
4,sample_000004,4rrk1/6pp/p1p1q3/3p2Q1/2PP2b1/1PR2N2/P5PP/5RK1...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1P/PPP1PPP...,r7k2r/pppp1pp/2n2n2/3p4/2B1P3/8/PPPP1P/PPP1PPP...,0.0,35,0.614035,0.0,0,0.695652,1,1,1,0


,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.9900,0.000617,0.994922,0.0175,0.043973,2.2775
1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.9850,0.000563,0.999727,0.0100,0.044427,2.3025
2,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.9775,0.000713,0.999648,0.0225,0.044630,2.3050
3,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.9750,0.000830,0.997031,0.0200,0.044088,2.2875
4,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.9700,0.000838,0.984766,0.0225,0.042988,2.2250
5,"Qwen + LoRA (Learned Reordering, frozen policy...","token reordering, 256px",0.9675,0.000881,0.994531,0.0225,0.043736,2.2625
6,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.9675,0.002206,0.992070,0.0200,0.043646,2.2600
7,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.9625,0.003055,0.996680,0.0125,0.046081,2.3850
8,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.9575,0.001550,0.996602,0.0175,0.044122,2.2950
9,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.9525,0.001315,0.977070,0.0250,0.043923,2.2800


## 5. LoRA baseline (raster order)

Supervised LoRA fine-tuning with the native raster order (= rank-wise). Up to 10 epochs with early stopping on validation loss; the best epoch is kept, also across a Colab disconnect.

The first cell trains the model, or loads it if the run is already finished on the Hub. The second cell evaluates the published adapter on the test set.

In [15]:
raster_repo = experiment_repo_id(TASK, "raster", TAG, hf_org_prefix)
trained = finetune_and_push_chessboard_model(
    "raster", dataset_task1, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
)
del trained
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK1 | STRATEGY: RASTER -> bdatm-project/qwen-task1-raster-tok256-lora


training_recipe.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Found a resumable checkpoint on the Hub for 'bdatm-project/qwen-task1-raster-tok256-lora': /root/.cache/huggingface/hub/models--bdatm-project--qwen-task1-raster-tok256-lora/snapshots/c2674a9ca1fc9d0bc6865ca3a5ee2c8838c219f3/last-checkpoint
Resuming from /root/.cache/huggingface/hub/models--bdatm-project--qwen-task1-raster-tok256-lora/snapshots/c2674a9ca1fc9d0bc6865ca3a5ee2c8838c219f3/last-checkpoint
Training for 'bdatm-project/qwen-task1-raster-tok256-lora' already finished on the Hub, skipping training and loading its final adapter.


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Nothing new to push for bdatm-project/qwen-task1-raster-tok256-lora.


In [16]:
raster_model = PeftModel.from_pretrained(load_base_model(), raster_repo)
raster_results_df, raster_summary_df = evaluate_with_order(
    TASK, raster_model, processor, dataset_task1["test"], hf_org_prefix=hf_org_prefix, adapter_repo_id=raster_repo,
    model_name=experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG), force=FORCE_EVAL,
)
all_models_results = append_or_replace(all_models_results, raster_summary_df)
display(headline_table(all_models_results, TASK))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task1] Using cached evaluation for 'Qwen + LoRA (Raster - SFT) [tok256]', skipping inference.


(…)lora_%28raster_-_sft%29_%5Btok256%5D.csv:   0%|          | 0.00/88.1k [00:00<?, ?B/s]

,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.9900,0.000617,0.994922,0.0175,0.043973,2.2775
1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.9850,0.000563,0.999727,0.0100,0.044427,2.3025
2,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.9775,0.000713,0.999648,0.0225,0.044630,2.3050
3,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.9750,0.000830,0.997031,0.0200,0.044088,2.2875
4,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.9700,0.000838,0.984766,0.0225,0.042988,2.2250
5,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.9675,0.002206,0.992070,0.0200,0.043646,2.2600
6,"Qwen + LoRA (Learned Reordering, frozen policy...","token reordering, 256px",0.9675,0.000881,0.994531,0.0225,0.043736,2.2625
7,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.9625,0.003055,0.996680,0.0125,0.046081,2.3850
8,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.9575,0.001550,0.996602,0.0175,0.044122,2.2950
9,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.9525,0.001315,0.977070,0.0250,0.043923,2.2800


## 6. Training-free reordering (TF)

The raster-trained LoRA model reads the *same* images with a different token order, with no retraining. Since the board is intact, this measures how much the model depends on the order it was trained with.

In [17]:
for strat in STRATEGIES:
    _, summary = evaluate_with_order(
        TASK, raster_model, processor, dataset_task1["test"], order=strat, hf_org_prefix=hf_org_prefix,
        adapter_repo_id=raster_repo, model_name=experiment_model_name(f"Qwen + LoRA Raster, read {strat} (TF)", TAG),
        force=FORCE_EVAL,
    )
    all_models_results = append_or_replace(all_models_results, summary)
display(headline_table(all_models_results, TASK))

[task1] Using cached evaluation for 'Qwen + LoRA Raster, read zigzag (TF) [tok256]', skipping inference.


(…)2C_read_zigzag_%28tf%29_%5Btok256%5D.csv:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

[task1] Using cached evaluation for 'Qwen + LoRA Raster, read spiral_center_out (TF) [tok256]', skipping inference.


(…)ral_center_out_%28tf%29_%5Btok256%5D.csv:   0%|          | 0.00/98.2k [00:00<?, ?B/s]

[task1] Using cached evaluation for 'Qwen + LoRA Raster, read file_wise (TF) [tok256]', skipping inference.


(…)read_file_wise_%28tf%29_%5Btok256%5D.csv:   0%|          | 0.00/98.1k [00:00<?, ?B/s]

[task1] Using cached evaluation for 'Qwen + LoRA Raster, read hilbert (TF) [tok256]', skipping inference.


(…)C_read_hilbert_%28tf%29_%5Btok256%5D.csv:   0%|          | 0.00/96.4k [00:00<?, ?B/s]

,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.9900,0.000617,0.994922,0.0175,0.043973,2.2775
1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.9850,0.000563,0.999727,0.0100,0.044427,2.3025
2,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.9775,0.000713,0.999648,0.0225,0.044630,2.3050
3,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.9750,0.000830,0.997031,0.0200,0.044088,2.2875
4,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.9700,0.000838,0.984766,0.0225,0.042988,2.2250
5,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.9675,0.002206,0.992070,0.0200,0.043646,2.2600
6,"Qwen + LoRA (Learned Reordering, frozen policy...","token reordering, 256px",0.9675,0.000881,0.994531,0.0225,0.043736,2.2625
7,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.9625,0.003055,0.996680,0.0125,0.046081,2.3850
8,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.9575,0.001550,0.996602,0.0175,0.044122,2.2950
9,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.9525,0.001315,0.977070,0.0250,0.043923,2.2800


## 7. Fine-tuned reordering (SFT)

One LoRA run per strategy, trained and evaluated with that token order. Each run loads a fresh base model and frees it afterwards.

In [18]:
for strat in STRATEGIES:
    trained = finetune_and_push_chessboard_model(
        strat, dataset_task1, processor, load_base_model(), peft_config, TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    )
    del trained
    free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


TASK: TASK1 | STRATEGY: ZIGZAG -> bdatm-project/qwen-task1-zigzag-tok256-lora


training_recipe.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Found a resumable checkpoint on the Hub for 'bdatm-project/qwen-task1-hilbert-tok256-lora': /root/.cache/huggingface/hub/models--bdatm-project--qwen-task1-hilbert-tok256-lora/snapshots/be413dcdd0753a90c8b447e6cc8199b27eb0bc87/last-checkpoint
Resuming from /root/.cache/huggingface/hub/models--bdatm-project--qwen-task1-hilbert-tok256-lora/snapshots/be413dcdd0753a90c8b447e6cc8199b27eb0bc87/last-checkpoint
Training for 'bdatm-project/qwen-task1-hilbert-tok256-lora' already finished on the Hub, skipping training and loading its final adapter.


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Nothing new to push for bdatm-project/qwen-task1-hilbert-tok256-lora.


In [19]:
for strat in STRATEGIES:
    repo = experiment_repo_id(TASK, strat, TAG, hf_org_prefix)
    strat_model = PeftModel.from_pretrained(load_base_model(), repo)
    _, summary = evaluate_with_order(
        TASK, strat_model, processor, dataset_task1["test"], order=strat, hf_org_prefix=hf_org_prefix, adapter_repo_id=repo,
        model_name=experiment_model_name(f"Qwen + LoRA ({strat.capitalize()} - SFT)", TAG), force=FORCE_EVAL,
    )
    all_models_results = append_or_replace(all_models_results, summary)
    del strat_model
    free_gpu()
display(headline_table(all_models_results, TASK))

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task1] Using cached evaluation for 'Qwen + LoRA (Zigzag - SFT) [tok256]', skipping inference.


(…)lora_%28zigzag_-_sft%29_%5Btok256%5D.csv:   0%|          | 0.00/88.0k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task1] Using cached evaluation for 'Qwen + LoRA (Spiral_center_out - SFT) [tok256]', skipping inference.


...

(…)a_%28file_wise_-_sft%29_%5Btok256%5D.csv:   0%|          | 0.00/88.1k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task1] Using cached evaluation for 'Qwen + LoRA (Hilbert - SFT) [tok256]', skipping inference.


(…)ora_%28hilbert_-_sft%29_%5Btok256%5D.csv:   0%|          | 0.00/88.1k [00:00<?, ?B/s]

,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.9900,0.000617,0.994922,0.0175,0.043973,2.2775
1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.9850,0.000563,0.999727,0.0100,0.044427,2.3025
2,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.9775,0.000713,0.999648,0.0225,0.044630,2.3050
3,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.9750,0.000830,0.997031,0.0200,0.044088,2.2875
4,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.9700,0.000838,0.984766,0.0225,0.042988,2.2250
5,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.9675,0.002206,0.992070,0.0200,0.043646,2.2600
6,"Qwen + LoRA (Learned Reordering, frozen policy...","token reordering, 256px",0.9675,0.000881,0.994531,0.0225,0.043736,2.2625
7,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.9625,0.003055,0.996680,0.0125,0.046081,2.3850
8,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.9575,0.001550,0.996602,0.0175,0.044122,2.2950
9,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.9525,0.001315,0.977070,0.0250,0.043923,2.2800


## 8. Learned reordering (REOrder)

REOrder's Plackett-Luce policy (`external/REOrder`) over the 64 squares, trained jointly with LoRA by REINFORCE (`src/training/learned_reordering.py`): one warm-up epoch in raster order, then the policy samples orders with a Gumbel temperature annealed 0.2 -> 0. Reward = minus the answer's cross-entropy; two orders are scored on each sample and compared with each other, so puzzle difficulty cancels out. Up to 10 epochs with early stopping on validation loss (patience 2); the best epoch is kept, together with the order it was validated with (the policy's greedy order, or raster if the warm-up epoch was best). Resumable from Hub checkpoints.

In [20]:
learned_model, learned_policy, learned_repo, learned_order = train_with_learned_reordering(
    dataset_task1, processor, load_base_model(), peft_config, task=TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
    num_train_epochs=10, policy_start_after_epochs=1,
)
print("Order used for evaluation (reading step of each square):")
print(describe_order(learned_order))
print("\nPolicy's greedy order:")
print(describe_order(greedy_order(learned_policy)))

_, learned_summary = evaluate_with_order(
    TASK, learned_model, processor, dataset_task1["test"], order=learned_order, hf_org_prefix=hf_org_prefix,
    adapter_repo_id=learned_repo, model_name=experiment_model_name("Qwen + LoRA (Learned Reordering)", TAG),
    force=FORCE_EVAL,
)
all_models_results = append_or_replace(all_models_results, learned_summary)
display(headline_table(all_models_results, TASK))
del learned_model
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[task1] Resuming learned reordering at epoch 6, row 0 (optimizer step 2400).
[task1] Early stopping: no improvement since epoch 3.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Order used for evaluation (reading step of each square):
8  0  1  2  3  4  5  6  8
7  7 10  9 11 12 14 13 15
6 16 17 18 19 20 22 21 23
5 24 25 26 27 29 28 30 31
4 32 34 35 33 37 36 38 39
3 40 43 42 41 44 46 45 47
2 48 49 51 50 52 53 55 54
1 58 56 57 59 60 61 63 62
   a  b  c  d  e  f  g  h

Policy's greedy order:
8  0  1  2  3  4  5  6  8
7  7 10  9 11 12 14 13 15
6 16 17 18 19 20 22 21 23
5 24 25 26 27 29 28 30 31
4 32 34 35 33 37 36 38 39
3 40 43 42 41 44 46 45 47
2 48 49 51 50 52 53 55 54
1 58 56 57 59 60 61 63 62
   a  b  c  d  e  f  g  h
[task1] Using cached evaluation for 'Qwen + LoRA (Learned Reordering) [tok256]', skipping inference.


(…)28learned_reordering%29_%5Btok256%5D.csv:   0%|          | 0.00/88.2k [00:00<?, ?B/s]

,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.9900,0.000617,0.994922,0.0175,0.043973,2.2775
1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.9850,0.000563,0.999727,0.0100,0.044427,2.3025
2,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.9775,0.000713,0.999648,0.0225,0.044630,2.3050
3,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.9750,0.000830,0.997031,0.0200,0.044088,2.2875
4,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.9700,0.000838,0.984766,0.0225,0.042988,2.2250
5,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.9675,0.002206,0.992070,0.0200,0.043646,2.2600
6,"Qwen + LoRA (Learned Reordering, frozen policy...","token reordering, 256px",0.9675,0.000881,0.994531,0.0225,0.043736,2.2625
7,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.9625,0.003055,0.996680,0.0125,0.046081,2.3850
8,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.9575,0.001550,0.996602,0.0175,0.044122,2.2950
9,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.9525,0.001315,0.977070,0.0250,0.043923,2.2800


### 8.1 Fine-tuning with the frozen policy (REOrder's last phase)

In REOrder, once the policy has been learned it is frozen: every batch uses its deterministic order (argsort of its logits, no more policy updates) and the same model keeps training, with validation in that order too. Here the LoRA adapter and the policy of the learned-reordering run are fetched from the Hub, the order is fixed to the policy's greedy order, and the model keeps training with the same Trainer recipe as the fixed orders (up to 10 epochs, early stopping, best epoch kept).

In [21]:
frozen_model, frozen_order, frozen_repo = finetune_with_frozen_policy(
    dataset_task1, processor, load_base_model(), task=TASK, hf_org_prefix=hf_org_prefix, tag=TAG,
)
print("Frozen policy order (reading step of each square):")
print(describe_order(frozen_order))

_, frozen_summary = evaluate_with_order(
    TASK, frozen_model, processor, dataset_task1["test"], order=frozen_order, hf_org_prefix=hf_org_prefix,
    adapter_repo_id=frozen_repo, model_name=experiment_model_name("Qwen + LoRA (Learned Reordering, frozen policy)", TAG),
    force=FORCE_EVAL,
)
all_models_results = append_or_replace(all_models_results, frozen_summary)
display(headline_table(all_models_results, TASK))
del frozen_model
free_gpu()

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

[task1] Frozen policy from bdatm-project/qwen-task1-learned-tok256-lora -> bdatm-project/qwen-task1-learned-frozen-tok256-lora


adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

training_recipe.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Frozen policy order (reading step of each square):
8  0  1  2  3  4  5  6  8
7  7 10  9 11 12 14 13 15
6 16 17 18 19 20 22 21 23
5 24 25 26 27 29 28 30 31
4 32 34 35 33 37 36 38 39
3 40 43 42 41 44 46 45 47
2 48 49 51 50 52 53 55 54
1 58 56 57 59 60 61 63 62
   a  b  c  d  e  f  g  h
[task1] Using cached evaluation for 'Qwen + LoRA (Learned Reordering, frozen policy) [tok256]', skipping inference.


(…)ing%2C_frozen_policy%29_%5Btok256%5D.csv:   0%|          | 0.00/88.0k [00:00<?, ?B/s]

,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.9900,0.000617,0.994922,0.0175,0.043973,2.2775
1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.9850,0.000563,0.999727,0.0100,0.044427,2.3025
2,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.9775,0.000713,0.999648,0.0225,0.044630,2.3050
3,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.9750,0.000830,0.997031,0.0200,0.044088,2.2875
4,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.9700,0.000838,0.984766,0.0225,0.042988,2.2250
5,"Qwen + LoRA (Learned Reordering, frozen policy...","token reordering, 256px",0.9675,0.000881,0.994531,0.0225,0.043736,2.2625
6,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.9675,0.002206,0.992070,0.0200,0.043646,2.2600
7,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.9625,0.003055,0.996680,0.0125,0.046081,2.3850
8,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.9575,0.001550,0.996602,0.0175,0.044122,2.2950
9,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.9525,0.001315,0.977070,0.0250,0.043923,2.2800


## 9. Resolution ablation

Same raster LoRA recipe at 512px (legacy run: 2x2 tokens per square) vs 256px (1 token per square).

The 512px row is the raster run of our first implementation (see "Legacy runs" in the README). With raster order that implementation does not change the image, so the two rows differ only in the resolution.

The next cell evaluates this 512px model (repository `qwen-task1-standard-lora`) with the current
code and a 512px processor; in raster order this gives the same predictions as the legacy code.
Like every other evaluation, it uses the cached result unless `FORCE_EVAL = True`.

To train the 512px model with the current code instead, see "Running at 512px" in Section 3. Our
512px models are evaluated again with the current code in `Reproduce_results.ipynb`.

In [22]:
LEGACY_RASTER_512 = "Qwen + LoRA Fine-Tuning (from HF)"   # raster LoRA of the first runs, 512px images
legacy_repo = f"{hf_org_prefix}/qwen-{TASK}-standard-lora"

processor_512 = AutoProcessor.from_pretrained(model_id)
configure_image_resolution(processor_512, 512)
legacy_model = PeftModel.from_pretrained(load_base_model(), legacy_repo)
_, legacy_summary = evaluate_with_order(
    TASK, legacy_model, processor_512, dataset_task1["test"], hf_org_prefix=hf_org_prefix,
    adapter_repo_id=legacy_repo, model_name=LEGACY_RASTER_512, force=FORCE_EVAL,
)
all_models_results = append_or_replace(all_models_results, legacy_summary)
del legacy_model
free_gpu()

rows = all_models_results["model_name"].isin([LEGACY_RASTER_512, experiment_model_name("Qwen + LoRA (Raster - SFT)", TAG)])
display(headline_table(all_models_results[rows], TASK))

[resolution] processor set to 512px: {'image_size': 512, 'patch_grid': (32, 32), 'llm_token_grid': (16, 16), 'tokens_per_square': 4}


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

[task1] Using cached evaluation for 'Qwen + LoRA Fine-Tuning (from HF)', skipping inference.


(…)n_%2B_lora_fine-tuning_%28from_hf%29.csv:   0%|          | 0.00/87.9k [00:00<?, ?B/s]

,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.99,0.000617,0.994922,0.0175,0.043973,2.2775
1,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.97,0.000838,0.984766,0.0225,0.042988,2.2250


## 10. Results table

Diagnostic columns are recomputed from the cached per-sample predictions (no inference), then the table is pushed to the Hub and reloaded from there. `headline_table` shows the main metrics first.

In [23]:
all_models_results = backfill_cached_metrics(
    all_models_results, TASK, fens=None, hf_org_prefix=hf_org_prefix
)
results_repo = f"{hf_org_prefix}/evaluation-results-{TASK}"
push_results_table(results_repo, all_models_results, commit_message=f"{TASK}: comparison table")
display(headline_table(load_results_table(results_repo), TASK))

vanilla_qwen2.5-vl_%28zero-shot%29.csv:   0%|          | 0.00/181k [00:00<?, ?B/s]

qwen_%2B_lora_%28zigzag_-_tf%29.csv:   0%|          | 0.00/84.1k [00:00<?, ?B/s]

qwen_%2B_lora_%28spiral_-_tf%29.csv:   0%|          | 0.00/84.3k [00:00<?, ?B/s]

qwen_%2B_lora_%28file_wise_-_tf%29.csv:   0%|          | 0.00/84.0k [00:00<?, ?B/s]

qwen_%2B_lora_%28zigzag_-_sft%29.csv:   0%|          | 0.00/82.2k [00:00<?, ?B/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 10.9kB / 10.9kB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


,model_name,setup,FEN Exact Match (board),CER (board),Square-by-Square Acc,FEN EM (full string),CER (full string),Levenshtein (full string)
0,Qwen + LoRA Fine-Tuning (from HF),"legacy: pixel-tile reordering, 512px",0.9900,0.000617,0.994922,0.0175,0.043973,2.2775
1,Qwen + LoRA (Spiral_center_out - SFT) [tok256],"token reordering, 256px",0.9850,0.000563,0.999727,0.0100,0.044427,2.3025
2,Qwen + LoRA (File_wise - SFT),"legacy: pixel-tile reordering, 512px",0.9775,0.000713,0.999648,0.0225,0.044630,2.3050
3,Qwen + LoRA (Zigzag - SFT) [tok256],"token reordering, 256px",0.9750,0.000830,0.997031,0.0200,0.044088,2.2875
4,Qwen + LoRA (Raster - SFT) [tok256],"token reordering, 256px",0.9700,0.000838,0.984766,0.0225,0.042988,2.2250
5,Qwen + LoRA (File_wise - SFT) [tok256],"token reordering, 256px",0.9675,0.002206,0.992070,0.0200,0.043646,2.2600
6,"Qwen + LoRA (Learned Reordering, frozen policy...","token reordering, 256px",0.9675,0.000881,0.994531,0.0225,0.043736,2.2625
7,Qwen + LoRA (Spiral - SFT),"legacy: pixel-tile reordering, 512px",0.9625,0.003055,0.996680,0.0125,0.046081,2.3850
8,Qwen + LoRA (Learned Reordering) [tok256],"token reordering, 256px",0.9575,0.001550,0.996602,0.0175,0.044122,2.2950
9,Qwen + LoRA (Hilbert - SFT) [tok256],"token reordering, 256px",0.9525,0.001315,0.977070,0.0250,0.043923,2.2800


## 11. Diagnostics

### 11.1 Training health

`training_health_report` per run: eval-loss history, discontinuities, recipe markers, best vs published epoch.

On Colab, `save_diagnostics` also saves the tables to Google Drive, so the cell asks for access to Drive.

In [25]:
if CONFIG["colab"]:
    from google.colab import drive
    drive.mount("/content/drive")   # save_diagnostics writes here, surviving a runtime reset

health = {r: training_health_report(r) for r in [experiment_repo_id(TASK, s, TAG, hf_org_prefix) for s in ["raster", *STRATEGIES, "learned"]]}
health_df = pd.DataFrame([
    {"repo_id": r, "n_problems": len(h["problems"]), "problems": " | ".join(h["problems"])} for r, h in health.items()
])
display(health_df)
save_diagnostics(health_df, f"training_health_{TAG}", TASK)

Mounted at /content/drive
=== bdatm-project/qwen-task1-raster-tok256-lora ===
  eval_loss by epoch: 1:0.177, 2:0.130, 3:0.136, 4:0.121, 5:0.115, 6:0.111, 7:0.111, 8:0.113, 9:0.121
  train loss first/last: 1.8475208282470703 / 0.08502525687217713
  - no problems detected
=== bdatm-project/qwen-task1-zigzag-tok256-lora ===
  eval_loss by epoch: 1:0.215, 2:0.158, 3:0.127, 4:0.122, 5:0.116, 6:0.113, 7:0.112, 8:0.115, 9:0.116
  train loss first/last: 1.8357120513916017 / 0.09278430342674256
  - no problems detected
=== bdatm-project/qwen-task1-spiral_center_out-tok256-lora ===
  eval_loss by epoch: 1:0.390, 2:0.162, 3:0.135, 4:0.126, 5:0.118, 6:0.117, 7:0.111, 8:0.111, 9:0.113
  train loss first/last: 1.8316490173339843 / 0.09471374750137329
  - no problems detected
=== bdatm-project/qwen-task1-file_wise-tok256-lora ===
  eval_loss by epoch: 1:0.272, 2:0.142, 3:0.137, 4:0.125, 5:0.121, 6:0.114, 7:0.109, 8:0.110, 9:0.110
  train loss first/last: 1.8382030487060548 / 0.09627113342285157
  - n

,repo_id,n_problems,problems
0,bdatm-project/qwen-task1-raster-tok256-lora,0,
1,bdatm-project/qwen-task1-zigzag-tok256-lora,0,
2,bdatm-project/qwen-task1-spiral_center_out-tok...,0,
3,bdatm-project/qwen-task1-file_wise-tok256-lora,0,
4,bdatm-project/qwen-task1-hilbert-tok256-lora,0,
5,bdatm-project/qwen-task1-learned-tok256-lora,0,


Saved diagnostics to /content/drive/MyDrive/bdatm_diagnostics/task1/training_health_tok256.csv


PosixPath('/content/drive/MyDrive/bdatm_diagnostics/task1/training_health_tok256.csv')

### 11.2 Error analysis (FEN errors)

Qualitative error analysis of the raster LoRA model: how many squares are wrong per board, which pieces get confused, and examples.

In [26]:
breakdown = fen_error_breakdown(raster_results_df)
display(breakdown["wrong_squares_per_board"])
display(breakdown["confusions"].head(15))
display(breakdown["examples"])
save_diagnostics(raster_results_df, f"raster_lora_per_sample_{TAG}", TASK)

,count
boards,
0,388
unparseable,6
1,6


,count
count,
R -> .,2
b -> r,2
B -> P,1
B -> R,1


,ground_truth,predicted,wrong_squares
0,8/6k1/3p2p1/pppp1PPp/2nP3N/2P5/PP2r3/2K4R,8/6k1/3p2p1/pppp2PPP/2nP3N/2P5/PP2r3/2K4R w - ...,unparseable
1,1r1q2k1/p7/5P1r/4p3/2PpB2P/P2Pn1P1/1P1K4/R1Q4R,1r1q2k1/p7/5P1r/4p3/2PpB2P/P2Pn1P1/1P1K4/R1Q5R...,unparseable
2,2kr3r/pppq4/2n5/4Pbp1/3PN1pb/2P5/PP2K1BB/R2Q3R,2kr3r/pppq4/2n5/4Pbp1/3PN1pb/2P5/PP2K1BR/R2Q3R,h2: B -> R
3,r5k1/ppp3bp/3p1q2/3P2p1/2Pn1Nb1/3Q2P1/PP3BBP/4...,r5k1/ppp3bp/3p1q2/3P2p1/2Pn1Nb1/3Q2P1/PP3BPP/4...,g2: B -> P
4,r1k5/8/3P2pp/2q1Bbb1/p1P5/P4Q2/4R1PP/7K,r1k5/8/3P2pp/2q1Bbf1/p1P5/P4Q2/4R1PP/7K b - - ...,unparseable


Saved diagnostics to /content/drive/MyDrive/bdatm_diagnostics/task1/raster_lora_per_sample_tok256.csv


PosixPath('/content/drive/MyDrive/bdatm_diagnostics/task1/raster_lora_per_sample_tok256.csv')